In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

class GestureNet(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        x = self.dropout(self.relu(self.layer1(x)))
        x = self.relu(self.layer2(x))
        return self.layer3(x)

df = pd.read_csv('gestures.csv', header=None)
X = df.iloc[:, 1:].values.astype('float32')
y_str = df.iloc[:, 0].values

encoder = LabelEncoder()
y = encoder.fit_transform(y_str)
num_classes = len(encoder.classes_)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# --- 3. TRAINING ---
model = GestureNet(63, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"Training on {num_classes} classes: {list(encoder.classes_)}")
for epoch in range(400):
    model.train()
    optimizer.zero_grad()
    outputs = model(torch.tensor(X_train))
    loss = criterion(outputs, torch.tensor(y_train).long())
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0: print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# --- 4. SAVE ---
torch.save({
    'model_state': model.state_dict(),
    'classes': encoder.classes_,
    'input_size': 63
}, 'gesture_model.pth')

Training on 8 classes: ['Down', 'Three_fingers', 'Thumbs_down', 'Thumbs_up', 'Up', 'fist', 'palm', nan]
Epoch 0, Loss: 2.0870
Epoch 100, Loss: 0.8723
Epoch 200, Loss: 0.2837
Epoch 300, Loss: 0.1391


In [5]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pickle

class GestureNet(nn.Module):
    def __init__(self, input_size, num_classes):
        super(GestureNet, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        x = self.dropout(self.relu(self.layer1(x)))
        x = self.relu(self.layer2(x))
        return self.layer3(x)


In [ ]:
import cv2
import mediapipe as mp
import torch
import torch.nn as nn
import numpy as np
import pyautogui
import pandas as pd
from collections import deque
import time


class GestureNet(nn.Module):
    def __init__(self, input_size, num_classes):
        super(GestureNet, self).__init__() 
        self.layer1 = nn.Linear(input_size, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        x = self.dropout(self.relu(self.layer1(x)))
        x = self.relu(self.layer2(x))
        return self.layer3(x)


checkpoint = torch.load('gesture_model.pth', weights_only=False)
raw_classes = checkpoint['classes']
classes = [str(c) if pd.notnull(c) else "none" for c in raw_classes]

model = GestureNet(63, len(classes))
model.load_state_dict(checkpoint['model_state'])
model.eval()


pyautogui.FAILSAFE = True
pyautogui.PAUSE = 0 
screen_w, screen_h = pyautogui.size()
gesture_buffer = deque(maxlen=5)


prev_x = 0
slide_cooldown = 0
SLIDE_THRESHOLD = 0.04 

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.7)
cap = cv2.VideoCapture(0)

print(f"System Active. Classes: {classes}")

while cap.isOpened():
    success, frame = cap.read()
    if not success: break
    
    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb_frame)
    
    if results.multi_hand_landmarks:
        lm = results.multi_hand_landmarks[0]
        
        raw_pts = []
        for p in lm.landmark: raw_pts.extend([p.x, p.y, p.z])
        wx, wy, wz = raw_pts[0], raw_pts[1], raw_pts[2]
        processed_pts = [(raw_pts[i]-wx) if i%3==0 else (raw_pts[i]-wy) if i%3==1 else (raw_pts[i]-wz) for i in range(63)]
        
        with torch.no_grad():
            inp = torch.tensor([processed_pts]).float()
            output = model(inp)
            gesture_idx = torch.argmax(output, dim=1).item()
            stable_gesture = classes[gesture_idx]
            gesture_buffer.append(stable_gesture)
        
        current_action = max(set(gesture_buffer), key=list(gesture_buffer).count)

        
        idx_tip = lm.landmark[8]
        
        if current_action in ['Up', 'palm']:
            tx = np.interp(idx_tip.x, [0.15, 0.85], [0, screen_w])
            ty = np.interp(idx_tip.y, [0.15, 0.85], [0, screen_h])
            pyautogui.moveTo(tx, ty, _pause=False)

        
        elif current_action == 'Three_fingers':
            curr_time = time.time()
            curr_x = idx_tip.x
            
            if curr_time - slide_cooldown > 0.6: 
                if prev_x != 0:
                    diff = curr_x - prev_x
                    if diff > SLIDE_THRESHOLD:
                        pyautogui.hotkey('ctrl', 'right')
                        slide_cooldown = curr_time
                    elif diff < -SLIDE_THRESHOLD: 
                        pyautogui.hotkey('ctrl', 'left')
                        slide_cooldown = curr_time
                prev_x = curr_x
        else:
            prev_x = 0 

        if current_action == 'fist':
            pyautogui.click()
            gesture_buffer.clear()
        elif current_action == 'Thumbs_up':
            pyautogui.press('volumeup')
        elif current_action == 'Thumbs_down':
            pyautogui.press('volumedown')
        elif current_action == 'Down':
            pyautogui.scroll(-10)

        # Visuals
        cv2.putText(frame, f"Action: {current_action}", (10, 50), 1, 2, (0, 255, 0), 2)
        mp.solutions.drawing_utils.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS)

    cv2.imshow('Hand Control', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

I0000 00:00:1770484383.237757       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


System Active. Classes: ['Down', 'Three_fingers', 'Thumbs_down', 'Thumbs_up', 'Up', 'fist', 'palm', 'none']


: 